# Transform constructors Data

1. Read bronze _constructors_ table
2. Keep only the columns required for analytics (Drop _url_ column).
3. Standardise column names using snake_case (_constructorId_ -> _constructor_id_ for example)
4. Rename columns to make them more meaningful (_name_ -> _constructor_name_ for example)
5. Remove duplicate records
6. Transform values of columns _nationality_ to Title Case
7. Write the transformed data to silver _constructors_ table.

In [0]:
%run ../00-common/01.environment-config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructors"
silver_table = f"{catalog_name}.{silver_schema}.constructors"

### Step 1 - Read bronze _constructors_ table

In [0]:
# Methode 1 that allows options since it's an API call:
# constructors_df = spark.read.table(bronze_table)

In [0]:
# Method 2 using a method with no additionnal options
constructors_df = spark.table(bronze_table)

In [0]:
display(constructors_df)

**2. Keep only the columns required for analytics (Drop _url_ column).**

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_dropped_df = constructors_df.drop("url")

In [0]:
display(constructors_dropped_df)

**3. Standardise column names using snake_case (_circuitId_ -> _circuit_id_ for example) &
4. Rename columns to make them more meaningful (_lat_ -> latitude for example)**

In [0]:
# ANCIENNE VERSION!!
# constructors_renamed_df = (
#     constructors_selected_df
#     .withColumnRenamed("circuitId", "circuit_id")
#     .withColumnRenamed("circuitName", "circuit_name")
#     .withColumnRenamed("lat", "latitude")
#     .withColumnRenamed("long","longitude")                       
# )

In [0]:
constructors_renamed_df = (
    constructors_dropped_df
    .withColumnsRenamed({
        "name": "constructor_name",
        "constructorId": "constructor_id"
    })                     
)

In [0]:
display(constructors_renamed_df)

**5. Remove duplicate records**

In [0]:
# constructors_distinct_df = constructors_valid_df.distinct()

In [0]:
constructors_distinct_df = constructors_renamed_df.dropDuplicates(["constructor_id"])

In [0]:
display(constructors_distinct_df)

**6. Transform values of column nationality to Title Case**

In [0]:
constructors_final_df = (
    constructors_distinct_df
        .withColumn("nationality", F.initcap(F.col("nationality")))
)

In [0]:
display(constructors_final_df)

**7. Write the transformed data to silver constructors table.**

In [0]:
(
    constructors_final_df
        .write
        .mode("overwrite")
        .format("delta")
        .saveAsTable(silver_table)
)

In [0]:
display(spark.table(silver_table))